# ASTR 457: Foundations of Data Science in Astronomy

**Fall 2026, University of Illinois Urbana-Champaign**

Prof. Gautham Narayan | TA: Abha Vishwakarma

Tue Sep 22, 2026 - Day 9: Regression I: least squares, and what it assumes

<img src="images/uiuc_logo.png" alt="University of Illinois Urbana-Champaign wordmark" width="220" style="display:block;margin:0 auto;">

## Today's plan: one paper, every regression problem

Hubble's 1929 paper is one straight line through 24 points. Every step of fitting it is a problem we still have, and he met them in this order:

1. a line through 24 galaxies: least squares, and the 3 things it assumes
2. he fit 4 parameters: the design matrix, and why it is the input to every machine-learning model
3. his distances came from Cepheids: today's Cepheids have error bars, and the weights matter (WLS)
4. some of the points are wrong: outliers, the residuals that show them, the mixture model from Day 3
5. his distances were the noisy variable: which variable has the error changes the slope (Thursday)

Recap first, with a real BIC call on a CO line.


## Quiz 2

* 15 minutes, on the sheet, closed everything
* 3 questions: an error bar and a reduced $\chi^2$, a bound, and a $\chi^2$ drop
* no computing beyond one division and one square
* hand it in when the timer goes; then we start

In [ ]:
# Presenter setup (a RISE "skip" cell: it runs, but is not a slide).
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy import stats

def load_csv(path):
    """genfromtxt with the '#' provenance header skipped; every data CSV in this week carries one."""
    with open(path) as _f:
        _nhead = sum(1 for _line in _f if _line.startswith('#'))
    return np.genfromtxt(path, delimiter=',', skip_header=_nhead, names=True, dtype=None, encoding='utf-8')

# Hubble 1929, PNAS 15, 168, Table 1: 24 nebulae, distance in Mpc and radial velocity in km/s (provenance in the header)
hub = load_csv('data/hubble1929_table1.csv')
r_hub, v_hub = hub['r_Mpc'].astype(float), hub['v_kms'].astype(float)

# Hogg, Bovy & Lang 2010, arXiv:1008.4686, Table 1: the 20-point exercise data with four planted outliers
hogg = load_csv('data/hogg2010_table1.csv')
x_h, y_h, sy_h = hogg['x'].astype(float), hogg['y'].astype(float), hogg['sigma_y'].astype(float)

print(f"Hubble: {len(r_hub)} nebulae; Hogg: {len(x_h)} points")

## Two questions from last class

* the two-Gaussian fit dropped $\chi^2$ from 1112 to 768. Why is that drop, by itself, not evidence for a second line? What is?
* AIC and BIC both kept falling from 2 Gaussians to 3.
* Did that mean 3 was right?
* What did the third one fit?

## Recap

* for nested models $\chi^2$ always goes down when you add parameters (it has to)
* so a lower $\chi^2$ by itself says nothing about whether the second line is real
* for nested models the likelihood-ratio test compares $\Delta\chi^2$ to a $\chi^2$ with $\Delta k$ degrees of freedom ($\Delta k$ = 3 here, the number of added parameters)
* AIC and BIC compare any two models, and only ever pick the best of the family you offered
* the residuals said the two-Gaussian model was wrong before the criteria did

In [ ]:
# Visual reminder: Thursday's one-, two- and three-Gaussian fits to the SDSS J1427+6354 H-alpha line, refit here (a "skip" cell)
def _load_thu():
    # Thursday's data and preprocessing (lecture/04/day8_lecture.ipynb, presenter setup), reproduced so the fits can be drawn again
    p = '../04/data/sdss_double_peaked_halpha.csv'
    with open(p) as f:
        nh = sum(1 for l in f if l.startswith('#'))
    d = np.genfromtxt(p, delimiter=',', skip_header=nh, names=True)
    w, fr, er = d['wave_rest_A'], d['flux'], d['flux_err']
    cw = ((w > 6200) & (w < 6350)) | ((w > 6750) & (w < 6900))
    A = np.vstack([np.ones(cw.sum()), w[cw] - 6550]).T * (1 / er[cw])[:, None]
    coef, *_ = np.linalg.lstsq(A, fr[cw] / er[cw], rcond=None)
    cov = np.linalg.inv(A.T @ A)
    cont = coef[0] + coef[1] * (w - 6550)
    cerr = np.sqrt(cov[0, 0] + cov[1, 1] * (w - 6550)**2 + 2 * cov[0, 1] * (w - 6550))
    fl = fr / cont
    el = fl * np.sqrt((er / fr)**2 + (cerr / cont)**2)
    use = np.ones_like(w, dtype=bool)
    for l in (6548.0, 6562.8, 6583.5, 6716.4, 6730.8):
        use &= ~((w > l - 8) & (w < l + 8))
    return w, fl, el, use

def _gauss(x, amp, cen, wid):
    return amp * np.exp(-0.5 * ((x - cen) / wid)**2)

def _fit_thu(w, fl, el, use, kmax=3):
    ww, ff, ee = w[use], fl[use], el[use]
    def chi2(p, k):
        if min(p[0::3]) < 0 or min(p[2::3]) < 1:
            return 1e30
        m = 1 + sum(_gauss(ww, *p[3*j:3*j+3]) for j in range(k))
        return np.sum((ff - m)**2 / ee**2)
    opts = {'xatol': 1e-7, 'fatol': 1e-7, 'maxiter': 40000, 'maxfev': 40000}
    fits = {}
    r = minimize(chi2, x0=[0.7, 6560, 60.0], args=(1,), method='Nelder-Mead', options=opts)
    fits[1] = minimize(chi2, x0=r.x, args=(1,), method='Nelder-Mead', options=opts).x
    r = minimize(chi2, x0=[0.6, 6510, 35.0, 0.5, 6640, 35.0], args=(2,), method='Nelder-Mead', options=opts)
    fits[2] = minimize(chi2, x0=r.x, args=(2,), method='Nelder-Mead', options=opts).x
    for k in range(3, kmax + 1):
        prev = fits[k - 1]
        m = 1 + sum(_gauss(ww, *prev[3*j:3*j+3]) for j in range(k - 1))
        x0 = list(prev) + [0.05, ww[np.argmax(np.abs(ff - m))], 20.0]
        fits[k] = minimize(chi2, x0=x0, args=(k,), method='Nelder-Mead',
                           options={'xatol': 1e-8, 'fatol': 1e-8, 'maxiter': 40000}).x
    return fits, {k: chi2(p, k) for k, p in fits.items()}, ww.size

w_thu, fl_thu, el_thu, use_thu = _load_thu()
fits_thu, chi2_thu, N_thu = _fit_thu(w_thu, fl_thu, el_thu, use_thu)
k_thu = np.array([3, 6, 9])
chi2_arr = np.array([chi2_thu[k] for k in (1, 2, 3)])
aic_thu = chi2_arr + 2 * k_thu
bic_thu = chi2_arr + k_thu * np.log(N_thu)

def plot_recap():
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.4), gridspec_kw={'width_ratios': [1.7, 1]})
    ax = axs[0]
    ax.errorbar(w_thu, fl_thu, yerr=el_thu, fmt='.', color='0.55', ms=2.5, alpha=0.5, elinewidth=0.4, label='SDSS J1427+6354')
    for k, col, ls in ((1, 'C0', '-'), (2, 'C2', '-'), (3, 'C3', '--')):
        p = fits_thu[k]
        m = 1 + sum(_gauss(w_thu, *p[3*j:3*j+3]) for j in range(k))
        ax.plot(w_thu, m, color=col, ls=ls, lw=1.6, label=fr'{k} Gaussian{"s" if k > 1 else ""}: $\chi^2$ = {chi2_thu[k]:.0f}')
    ax.set_xlim(6250, 6900); ax.set_ylim(0.85, 2.3)
    ax.set_xlabel(r'rest wavelength [$\AA$]'); ax.set_ylabel('flux / continuum')
    ax.set_title(r"Thursday's H$\alpha$ line, the three fits", fontsize=10)
    ax.legend(frameon=False, fontsize=8, loc='upper right')
    ax = axs[1]
    ncomp = [1, 2, 3]
    ax.plot(ncomp, chi2_arr, 'o-', color='0.4', label=r'$\chi^2$')
    ax.plot(ncomp, aic_thu, 's-', color='C0', label='AIC')
    ax.plot(ncomp, bic_thu, '^-', color='C3', label='BIC')
    ax.set_xticks(ncomp); ax.set_xlabel('number of Gaussians'); ax.set_ylabel('lower is better')
    ax.set_title('the criteria keep falling', fontsize=10)
    ax.legend(frameon=False, fontsize=8)
    plt.tight_layout()
    plt.show()
    print(f"N = {N_thu} pixels; chi2 = " + ", ".join(f"{chi2_thu[k]:.1f}" for k in (1, 2, 3)) + " for k = 3, 6, 9 (Thursday: 1112.3, 767.6, 695.4)")


In [ ]:
# Thursday's three fits drawn on the line again, and the criteria beside them: they never stop falling because the family (sums of Gaussians) was wrong
plot_recap()


## A real AIC/BIC call: one CO line or two?

<table style="border:none; margin:0 auto;"><tr style="border:none;"><td style="border:none; padding:4px; vertical-align:top;"><img src="images/martin2024_fig2_w0134_single_gaussian_co.png" alt="CO(4-3) emission line spectrum of the hot dust-obscured galaxy W0134-2922, flux density against offset velocity, with a single Gaussian fit (blue dashed curve) tracing the one noisy but clearly single-peaked emission bump near zero velocity" width="520" style="display:block;margin:0 auto;"></td></tr></table>

<div style="font-size:0.8em; text-align:center; color:#666;">Martin, Blain, Díaz-Santos, Assef, Tsai, Jun, Eisenhardt, Wu, Vayner &amp; Fernández Aranda 2024, MNRAS (arXiv:2409.11013), Fig. 2, CC BY 4.0. W0134-2922, a single-Gaussian fit.</div>

* CO is carbon monoxide, the molecule we use to find cold gas (tens of kelvin, far too cold to glow in visible light)
* its rotational lines are at mm wavelengths, and this is an ALMA spectrum (the Atacama Large Millimeter Array, in Chile)
* the $x$ axis is velocity.
* Every wavelength shift is read as a Doppler shift, and the profile is a histogram of gas velocities
* 1 Gaussian = gas at one bulk velocity; 2 = two clumps, a rotating disc, or an outflow, the same question as the H$\alpha$ line last class
* W0134 and W2305 are hot dust-obscured galaxies (hot DOGs), quasars (the brightest AGN) so buried in dust that they shine mostly in the infrared
* W0134-2922, above, is 1 Gaussian, and a tentative detection (S/N $<$ 3)


## $\Delta$BIC = 15.8: what the number means

<table style="border:none; margin:0 auto;"><tr style="border:none;"><td style="border:none; padding:4px; vertical-align:top;"><img src="images/martin2024_fig2_w2305_double_gaussian_co.png" alt="CO(4-3) emission line spectrum of the hot dust-obscured galaxy W2305-0039, flux density against offset velocity, decomposed into two Gaussian components (magenta and black dashed curves) that sum to the blue dashed curve: a narrow bright component sitting on top of a broader, fainter one, together spanning a wider base than either alone" width="520" style="display:block;margin:0 auto;"></td></tr></table>

<div style="font-size:0.8em; text-align:center; color:#666;">Martin et al. 2024, MNRAS (arXiv:2409.11013), Fig. 2, CC BY 4.0: W2305-0039, a two-Gaussian fit with each component drawn separately.</div>

* W2305-0039: BIC $-26.1 \to -41.9$, $\Delta$BIC $= 15.8$, "very strong evidence" for the second component
* the BICs are negative because they keep the $\sum\ln(2\pi\sigma_i^2)$ constant that $\chi^2$ drops; only the difference $\Delta$BIC is comparable
* same scale as last class: $\Delta$BIC above 10 is "very strong" (Kass &amp; Raftery 1995)
* 4 of the 10 sources clear $\Delta$BIC $\geq 10$ against 1 Gaussian, the rest don't.
* BIC tells us which lines have two components
* Next: a likelihood whose MLE has a closed-form solution.
* One matrix inverse gives it, and 3 assumptions buy it

## Hubble 1929

<img src="images/hubble1929_fig1_velocity_distance.png" alt="Hubble's 1929 Figure 1: radial velocity in km/s against distance in parsecs for 24 extragalactic nebulae, filled points and a solid line for the individual nebulae, open points and a dashed line for the same nebulae grouped by proximity" style="display:block;margin:0 auto;max-height:400px">

<div style="font-size:0.8em; text-align:center; color:#666;">Hubble 1929, PNAS 15, 168, Fig. 1. Public domain. The $y$-axis label reads km; the units are km/s.</div>

* 24 "nebulae": galaxies, though in 1929 that was still being argued
* Vesto Slipher measured 20 of the 24 velocities at Lowell Observatory; the distances came from Cepheids and the brightest stars (assumed equally bright everywhere)
* velocity is proportional to distance, $v = K r$.
* This is why we know the Universe is expanding
* Hubble got $K = 465 \pm 50$ km/s per Mpc (a megaparsec, 3.26 million light years) and adopted 500.
* $K$ is what we now call $H_0$
* today's value is ~70.
* His distances were ~7$\times$ too small (remember the Cepheid calibration), and $K = v/r$ came out ~7$\times$ too big
* open circles: the same 24 averaged in 9 groups of sky neighbours, taken to sit at one distance.
* The scatter drops and $K = 513$
* both lines are least-squares fits, today's subject.
* NB his velocities have the Sun's motion removed; the raw table we plot next does not


## Hubble's Table 1

* the same 24 galaxies, straight from his table, distance in Mpc against velocity in km/s
* no error bars in the table (in 1929 the distance was the uncertain number, and nobody could quantify by how much)
* this is the dataset we fit for the next 20 minutes

In [ ]:
# Hubble 1929 Table 1: velocity against distance, nothing fitted yet (a "skip" cell)
def plot_hubble_raw():
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.plot(r_hub, v_hub, 'o', color='C0', ms=6)
    ax.axhline(0, color='0.6', lw=0.8)
    ax.set_xlabel('distance $r$ [Mpc]')
    ax.set_ylabel('radial velocity $v$ [km/s]')
    ax.set_title("Hubble 1929, Table 1: 24 nebulae")
    plt.tight_layout()
    plt.show()

In [ ]:
# the 24 nebulae, as Hubble tabulated them
plot_hubble_raw()

* a trend, a lot of scatter, and 5 galaxies coming toward us (negative $v$)
* you would all draw a line through this.
* Which line, and what does it assume?

## Least squares as maximum likelihood

*Hubble's problem 1 of 5: which line, and what it assumes.*

Model $v_i = K r_i + v_0$ ($v_0$ the intercept), Gaussian noise $\sigma$ on $v$ only, $r$ exact, points independent. Then the Day 5 likelihood is

$$-2\ln\mathcal{L} = \chi^2 + \text{const}, \qquad \chi^2 = \sum_i \frac{(v_i - K r_i - v_0)^2}{\sigma^2}$$

* NB: maximizing a Gaussian likelihood is equivalent to minimizing $\chi^2$.
* This is called "least squares"
* with all the $\sigma$ equal it is "ordinary least squares" (OLS)
* 3 assumptions, the same 3 as the straight line on Day 2 (Gaussian noise, $r$ exact, independent points)
* last week you minimized numerically.
* A line is linear in its parameters, and this one has a closed-form solution!
* we keep an intercept $v_0$ for now as a check.
* The physics says it should be 0, and the fit will tell us

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Least squares is maximum likelihood with Gaussian errors on $y$ alone.**

</div>

## Why a matrix?

*Hubble's problem 2 of 5: he fit 4 parameters, not 2.*

* $K$, plus the 3 components of the Sun's own velocity
* minimizing $\chi^2$ means one partial derivative per parameter, i.e. 4 equations, each linear in all 4 unknowns
* 4 linear equations in 4 unknowns is a matrix problem.
* Write it as one, and the solution is the same one line for 2 parameters or 200
* the matrix of known functions of the data is called the **design matrix**: one row per data point, one column per parameter
* that table is the `X` every `scikit-learn` model takes, and a neural network is the same picture with the columns learned (both in November)
* we build it for 2 columns first (a line with an intercept) so you can see the machinery, then for Hubble's 4


## The design matrix

Hubble's linear model, one equation per nebula, with $\epsilon_i$ the noise on that velocity:

$$v_i = v_0 + K\,r_i + \epsilon_i, \qquad i = 1, \ldots, 24$$

Stack the 24 equations. The design matrix $A$ is the table of known functions of the data, one row per nebula and one column per parameter:

$$\mathbf{v} = A\,\boldsymbol{\theta} + \boldsymbol{\epsilon}, \qquad A = \begin{pmatrix} 1 & r_1 \\ 1 & r_2 \\ \vdots & \vdots \\ 1 & r_{24} \end{pmatrix}, \quad \boldsymbol{\theta} = \begin{pmatrix} v_0 \\ K \end{pmatrix}$$

Take the partial derivatives of $\chi^2$ and set them to zero (just the usual minimization):

$$\frac{\partial\chi^2}{\partial\boldsymbol{\theta}} = -\frac{2}{\sigma^2}\,A^{\mathsf T}(\mathbf{v} - A\boldsymbol{\theta}) = 0 \quad\Longrightarrow\quad A^{\mathsf T}A\,\boldsymbol{\theta} = A^{\mathsf T}\mathbf{v}$$

* this linear system is called the **normal equations**; $A^{\mathsf T}$ is the transpose (rows become columns)
* $A$ is the design matrix here; last class it was a Gaussian amplitude
* solving it gives the estimate, $\hat{\boldsymbol{\theta}} = (A^{\mathsf T}A)^{-1} A^{\mathsf T}\mathbf{v}$

## The normal equations

For a line the matrix is 5 sums, and the error bars come from the curvature of the likelihood (remember Days 5 and 6):

$$A^{\mathsf T}A = \begin{pmatrix} N & \sum r_i \\ \sum r_i & \sum r_i^2 \end{pmatrix}, \qquad A^{\mathsf T}\mathbf{v} = \begin{pmatrix} \sum v_i \\ \sum r_i v_i \end{pmatrix}, \qquad \frac{\partial^2(-\ln\mathcal{L})}{\partial\boldsymbol{\theta}^2} = \frac{A^{\mathsf T}A}{\sigma^2} \;\Longrightarrow\; \mathrm{Cov}(\hat{\boldsymbol\theta}) = \sigma^2 (A^{\mathsf T}A)^{-1}$$

* the curvature of $-\ln\mathcal{L}$ is the inverse covariance, and $\chi^2 = -2\ln\mathcal{L}$ has the extra factor of 2
* Hubble's table has no error bars, so estimate $\sigma$ from the residuals: $s^2 = \mathrm{RSS}/(N - k)$, RSS the residual sum of squares
* $N - k$ is the $N - 1$ you remember from the sample variance, with one degree of freedom lost per fitted parameter ($k$ columns of $A$)
* Lab 04 Part 1 has you write this yourself

In [ ]:
# Ordinary least squares on Hubble's 24 nebulae, by the normal equations (a "skip" cell)
def ols(A, y):
    """theta_hat, its covariance (residual-scaled) and the residuals, for design matrix A and data y."""
    AtA_inv = np.linalg.inv(A.T @ A)
    theta = AtA_inv @ A.T @ y
    resid = y - A @ theta
    s2 = resid @ resid / (len(y) - A.shape[1])
    return theta, s2 * AtA_inv, resid

A_hub = np.vstack([np.ones_like(r_hub), r_hub]).T
theta_hub, cov_hub, res_hub = ols(A_hub, v_hub)
v0_hat, K_hat = theta_hub
sig_v0, sig_K = np.sqrt(np.diag(cov_hub))
# through the origin, as Hubble's headline number: one column, no intercept
K0_hat, cov_K0, _ = ols(r_hub[:, None], v_hub)
K_hubble = 465.0   # Hubble 1929, p. 170: the 24-nebula solution, which also solved for the Sun's motion (3 more columns), km/s/Mpc

def plot_hubble_ols():
    fig, (ax, ar) = plt.subplots(2, 1, figsize=(7, 4.6), sharex=True, gridspec_kw=dict(height_ratios=[3, 1.2], hspace=0.05))
    rr = np.linspace(0, 2.1, 20)
    ax.plot(r_hub, v_hub, 'o', color='C0', ms=6, label='Hubble 1929, Table 1')
    ax.plot(rr, v0_hat + K_hat * rr, 'C3', lw=1.8, label=fr'OLS: $K$ = {K_hat:.0f} $\pm$ {sig_K:.0f}, $v_0$ = {v0_hat:.0f} $\pm$ {sig_v0:.0f}')
    ax.plot(rr, K_hubble * rr, '0.4', ls='--', lw=1.2, label=fr'Hubble: $K$ = {K_hubble:.0f} through the origin')
    ax.set_ylabel('$v$ [km/s]'); ax.legend(frameon=False, fontsize=9)
    ar.axhline(0, color='k', lw=0.6)
    ar.plot(r_hub, res_hub, 'o', color='C0', ms=5)
    ar.set_ylabel('residual [km/s]'); ar.set_xlabel('distance $r$ [Mpc]')
    plt.show()
    print(f"OLS with intercept: K = {K_hat:.0f} +/- {sig_K:.0f} km/s/Mpc, v0 = {v0_hat:.0f} +/- {sig_v0:.0f} km/s, s = {np.sqrt(res_hub @ res_hub / 22):.0f} km/s")
    print(f"OLS through the origin: K = {K0_hat[0]:.0f} +/- {np.sqrt(cov_K0[0, 0]):.0f} km/s/Mpc   (Hubble 1929: {K_hubble:.0f} +/- 50)")

In [ ]:
# the fit drawn on the data it was fit to, Hubble's own slope dashed, residuals below
plot_hubble_ols()

* the two-column $K$, $454 \pm 75$, falls inside Hubble's $465 \pm 50$
* $s = \sqrt{s^2} \approx 230$ km/s per galaxy.
* With no error bars in the table, this is our only noise estimate
* $v_0 = -41 \pm 83$ km/s, i.e. consistent with 0, as the physics says it should be
* the residuals show no trend, and a line is fine.
* Whether $K$ is *right* is a different question (the distances)

## Linear in the parameters

* the columns of $A$ can be any known function of the data: $1$, $r$, $r^2$, $\sin\omega t$ (for a known frequency $\omega$), a template spectrum
* $y = \theta_0 + \theta_1 x + \theta_2 x^2$ is linear regression.
* $y = \theta_0\,e^{-\theta_1 x}$ is not: non-linear least squares has no closed form, and you solve it numerically (Day 7)
* a sinusoid at a known frequency is linear too, with $\sin\omega t$ and $\cos\omega t$ as columns
* the periodogram we mentioned last week is this fit, one frequency at a time (October)
* more columns always lower $\chi^2$ (the elephant from last class!).
* The likelihood-ratio test and BIC tell you when to stop
* `np.polyfit` and `np.linalg.lstsq` solve this same problem (by a matrix method that avoids inverting $A^{\mathsf T}A$, which is numerically safer)

In [ ]:
# Hubble's actual 1929 fit: r K + X cos(a) cos(d) + Y sin(a) cos(d) + Z sin(d) = v, four columns, no intercept (a "skip" cell)
coords = load_csv('data/hubble1929_coords.csv')
ra, dec = np.radians(coords['ra_deg'].astype(float)), np.radians(coords['dec_deg'].astype(float))
A_sun = np.vstack([r_hub, np.cos(ra) * np.cos(dec), np.sin(ra) * np.cos(dec), np.sin(dec)]).T
theta_sun, cov_sun, res_sun = ols(A_sun, v_hub)
K_sun, X_sun, Y_sun, Z_sun = theta_sun
sig_K_sun = np.sqrt(cov_sun[0, 0])
V_sun = np.sqrt(X_sun**2 + Y_sun**2 + Z_sun**2)
v_corr = v_hub - A_sun[:, 1:] @ theta_sun[1:]           # velocities with the Sun's motion removed

def plot_design_columns():
    """Hubble's 24 x 4 design matrix, printed: the first 5 rows, a row of dots, the last row, on the column colours."""
    show = [0, 1, 2, 3, 4, None, 23]
    fig, ax = plt.subplots(figsize=(7.2, 3.6))
    scaled = A_sun / np.abs(A_sun).max(axis=0)
    img = np.array([scaled[r] if r is not None else np.zeros(4) for r in show])
    ax.imshow(img, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1, alpha=0.55)
    for k, r in enumerate(show):
        for col in range(4):
            txt = r'$\vdots$' if r is None else f'{A_sun[r, col]:.2f}'
            ax.text(col, k, txt, ha='center', va='center', fontsize=12)
    ax.set_xticks(range(4)); ax.set_xticklabels(['$r$ [Mpc]', r'$\cos\alpha\cos\delta$', r'$\sin\alpha\cos\delta$', r'$\sin\delta$'], fontsize=11)
    ax.xaxis.tick_top()
    ax.set_yticks(range(len(show))); ax.set_yticklabels([('galaxy %d' % (r + 1)) if r is not None else '' for r in show], fontsize=10)
    ax.set_title("Hubble's design matrix $A$: 24 rows, 4 columns (colour = each column scaled to its largest entry)", fontsize=10, pad=28)
    plt.tight_layout()
    plt.show()

def plot_sun_geometry(alpha_deg=40., delta_deg=35.):
    """Left: RA and Dec on the sphere give the unit vector to a nebula. Right: the Sun's velocity projected on it."""
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
    fig = plt.figure(figsize=(10.5, 4.6))
    # ---- left: the sphere, the two angles, the unit vector ----
    ax = fig.add_subplot(121, projection='3d')
    u = np.linspace(0, 2 * np.pi, 73); vv = np.linspace(0, np.pi, 37)
    ax.plot_wireframe(np.outer(np.cos(u), np.sin(vv)), np.outer(np.sin(u), np.sin(vv)), np.outer(np.ones_like(u), np.cos(vv)),
                      color='0.88', lw=0.35, rstride=12, cstride=6)
    ax.plot(np.cos(u), np.sin(u), 0 * u, color='0.4', lw=1.3)
    ax.text(-0.2, -1.25, -0.25, 'celestial equator', fontsize=8.5, color='0.4')
    ax.quiver(0, 0, 0, 1.45, 0, 0, color='0.3', arrow_length_ratio=0.06, lw=0.9)
    ax.text(1.5, -0.05, 0.1, r'$\alpha$ = 0, $\delta$ = 0', fontsize=8.5, color='0.3')
    ax.quiver(0, 0, 0, 0, 0, 1.35, color='0.3', arrow_length_ratio=0.06, lw=0.9)
    ax.text(0.04, 0, 1.42, r'north pole, $\delta$ = 90$^\circ$', fontsize=8.5, color='0.3')
    a, d = np.radians(alpha_deg), np.radians(delta_deg)
    n = np.array([np.cos(a) * np.cos(d), np.sin(a) * np.cos(d), np.sin(d)])
    ax.quiver(0, 0, 0, *(1.25 * n), color='C0', lw=2.6, arrow_length_ratio=0.08)
    ax.text(*(1.3 * n + np.array([0, 0, 0.08])), 'to the nebula, ' + r'$\hat n$', fontsize=10, color='C0')
    aa = np.linspace(0, a, 30)
    ax.plot(np.cos(aa), np.sin(aa), 0 * aa, color='C1', lw=3)
    ax.text(np.cos(a / 2) * 1.1, np.sin(a / 2) * 1.1, -0.45, r'RA $\alpha$', fontsize=10, color='C1')
    dd = np.linspace(0, d, 30)
    ax.plot(np.cos(a) * np.cos(dd), np.sin(a) * np.cos(dd), np.sin(dd), color='C2', lw=3)
    ax.text(np.cos(a) * np.cos(d / 2) * 1.12, np.sin(a) * np.cos(d / 2) * 1.12, np.sin(d / 2) + 0.05, r'Dec $\delta$', fontsize=10, color='C2')
    ax.set_xlim(-0.8, 1.2); ax.set_ylim(-0.8, 1.2); ax.set_zlim(-0.8, 1.2); ax.set_box_aspect((1, 1, 1))
    ax.set_axis_off(); ax.view_init(elev=18, azim=-50)
    ax.set_title(r'right ascension $\alpha$ (longitude) and declination $\delta$ (latitude) fix the direction' + '\n'
                 + r'$\hat n = (\cos\alpha\cos\delta,\ \sin\alpha\cos\delta,\ \sin\delta)$', fontsize=9.5)
    # ---- right: the projection, in the plane of V and n ----
    ax2 = fig.add_subplot(122)
    n2 = np.array([np.cos(np.radians(28)), np.sin(np.radians(28))])
    V2 = 1.6 * np.array([np.cos(np.radians(78)), np.sin(np.radians(78))])
    proj = (V2 @ n2) * n2
    ax2.plot(0, 0, 'o', color='gold', ms=14, mec='k'); ax2.text(-0.12, -0.2, 'Sun', fontsize=10, ha='center')
    ax2.annotate('', xy=2.2 * n2, xytext=(0, 0), arrowprops=dict(arrowstyle='-|>', color='C0', lw=2.4))
    ax2.text(*(2.25 * n2), r'to the nebula, $\hat n$', color='C0', fontsize=10)
    ax2.annotate('', xy=V2, xytext=(0, 0), arrowprops=dict(arrowstyle='-|>', color='C3', lw=2.4))
    ax2.text(V2[0] - 0.05, V2[1] + 0.08, r"the Sun's velocity $\vec V_\odot$ = ($X$, $Y$, $Z$)", color='C3', fontsize=10, ha='center')
    ax2.plot([V2[0], proj[0]], [V2[1], proj[1]], ls=':', color='0.4', lw=1.2)
    ax2.annotate('', xy=proj, xytext=(0, 0), arrowprops=dict(arrowstyle='-|>', color='C3', lw=5, alpha=0.5))
    ax2.text(proj[0] + 0.1, proj[1] - 0.45, r'$\vec V_\odot\cdot\hat n$' + '\nthe part that shows up\nin the measured velocity', color='C3', fontsize=9.5)
    ax2.set_xlim(-0.9, 2.9); ax2.set_ylim(-0.6, 1.9); ax2.set_aspect('equal'); ax2.set_axis_off()
    ax2.set_title(r'the spectrum measures $v = K r + \vec V_\odot\cdot\hat n$' + '\n'
                  + r'$\vec V_\odot\cdot\hat n = X\cos\alpha\cos\delta + Y\sin\alpha\cos\delta + Z\sin\delta$: linear in $X, Y, Z$', fontsize=9.5)
    plt.tight_layout()
    plt.show()

def plot_hubble_four_columns():
    fig, ay = plt.subplots(figsize=(7, 4.2))
    rr = np.linspace(0, 2.1, 20)
    ay.plot(r_hub, v_hub, 'o', color='0.6', ms=5, label='as observed')
    ay.plot(r_hub, v_corr, 'o', color='C0', ms=6, label="with the Sun's motion removed")
    ay.plot(rr, K_sun * rr, 'C3', lw=1.8, label=fr'four columns: $K$ = {K_sun:.0f} $\pm$ {sig_K_sun:.0f}')
    ay.plot(rr, K_hubble * rr, '0.4', ls='--', lw=1.2, label=r'Hubble 1929: $K$ = 465 $\pm$ 50')
    ay.set_xlabel('distance $r$ [Mpc]'); ay.set_ylabel('$v$ [km/s]'); ay.legend(frameon=False, fontsize=8)
    plt.tight_layout()
    plt.show()
    print(f"four columns: K = {K_sun:.0f} +/- {sig_K_sun:.0f} km/s/Mpc; solar motion (X, Y, Z) = ({X_sun:.0f}, {Y_sun:.0f}, {Z_sun:.0f}) km/s, "
          f"|V_sun| = {V_sun:.0f} km/s   (Hubble: K = 465 +/- 50, X = -65, Y = +226, Z = -195, V = 306)")

## Where the Sun's motion goes

* the Sun is not at rest.
* Its own velocity adds a piece to every measured $v$
* the piece differs for every galaxy, because it depends on where on the sky the galaxy is
* a galaxy straight ahead of the Sun's motion gets the full speed added; one at right angles gets none
* i.e. a dot product, and a dot product is linear in $(X, Y, Z)$


In [ ]:
# the geometry behind the three solar-motion columns: RA and Dec on the sphere, the Sun's velocity, and its projection on one nebula
plot_sun_geometry()


## Hubble's four columns

Every velocity Hubble measured is the nebula's motion plus the Sun's own motion projected on the line of sight to it. A nebula's direction on the sky is its right ascension $\alpha$ (longitude) and declination $\delta$ (latitude), and the unit vector pointing at it is $\hat n = (\cos\alpha\cos\delta,\ \sin\alpha\cos\delta,\ \sin\delta)$. If the Sun moves at $(X, Y, Z)$, the spectrum sees $\vec V_\odot\cdot\hat n$ added to $Kr$:

$$v_i = K\,r_i + X\cos\alpha_i\cos\delta_i + Y\sin\alpha_i\cos\delta_i + Z\sin\delta_i + \epsilon_i$$

* the 3 trig terms are the components of $\hat n_i$, known numbers for each galaxy: 3 more columns of $A$, linear in $X, Y, Z$
* the intercept goes, because $v = Kr$ through the origin is the physics
* 4 columns and 24 rows.
* Look at the matrix, then we redo his fit


In [ ]:
# the real design matrix: Hubble's four columns, one row per nebula, the entries printed (the fit itself is next)
plot_design_columns()


In [ ]:
# his fit, redone from his table and the nebulae's sky positions: the Sun's motion removed, and K
plot_hubble_four_columns()

* same matrix, same inverse, 3 new columns
* $K = 465 \pm 56$ km/s/Mpc and a solar motion of $(-69, 235, -200)$ km/s.
* Hubble's 1929 numbers were $465 \pm 50$ and $(-65, 226, -195)$
* from his table and today's (J2000) coordinates; the sky grid drifts ~1$^\circ$ a century, a small correction to the 3 trig columns


<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Linear regression is any model that is a weighted sum of known shapes. The weights are the parameters.**

</div>

## Weighted least squares (WLS)

*Hubble's problem 3 of 5: his $K$ was only as good as his distances, and his table had no error bars. Today's data do.*

If the measurement errors are not all equal, the line must be closer to the points with lower measurement error. Modify the least-squares distance accordingly, and the previous derivation carries through with $\Sigma = \mathrm{diag}(\sigma_i^2)$, the covariance matrix with the error bars squared on its diagonal:

$$\chi^2 = \sum_i \frac{(y_i - A_i\boldsymbol{\theta})^2}{\sigma_i^2}, \qquad \hat{\boldsymbol{\theta}} = (A^{\mathsf T}\Sigma^{-1}A)^{-1}A^{\mathsf T}\Sigma^{-1}\mathbf{y}, \qquad \mathrm{Cov}(\hat{\boldsymbol{\theta}}) = (A^{\mathsf T}\Sigma^{-1}A)^{-1}$$

* $A_i$ is row $i$ of $A$.
* A point with twice the error bar counts a quarter as much
* the $\sigma_i$ are given to us in the data, and there is no $s^2$ to estimate
* the covariance comes from the error bars alone, if we believe them (the Cepheids, next)
* $A^{\mathsf T}\Sigma^{-1}A$ is the Fisher information matrix from Days 5 and 6, and its inverse is the covariance

In [ ]:
# OLS and WLS on the Riess et al. 2011 Cepheids in NGC 4258, the anchor galaxy of the distance ladder (a "skip" cell)
r11 = load_csv('data/r11_cepheids.csv')
sel = r11['gal'].astype(int) == 4258
logP_r11 = np.log10(r11['P'].astype(float)[sel])
m_r11, em_r11 = r11['m'].astype(float)[sel], r11['merr'].astype(float)[sel]
rej_r11 = np.char.startswith(r11['rej_flag'].astype(str)[sel], 'rej')   # R11 Table 2 flag: their own outlier rejection

def wls(A, y, sigma):
    """theta_hat and its covariance from the error bars, for diagonal Sigma."""
    W = 1.0 / sigma**2
    AtWA_inv = np.linalg.inv(A.T @ (A * W[:, None]))
    theta = AtWA_inv @ A.T @ (W * y)
    return theta, AtWA_inv

A_r11 = np.vstack([np.ones_like(logP_r11), logP_r11 - 1.0]).T          # pivot at P = 10 d
th_o, cov_o, res_o = ols(A_r11, m_r11)
th_w, cov_w = wls(A_r11, m_r11, em_r11)
chi2_w = np.sum(((m_r11 - A_r11 @ th_w) / em_r11)**2)
nu_w = m_r11.size - 2
z_r11 = (m_r11 - A_r11 @ th_w) / em_r11                                  # normalized residuals of the WLS fit
th_wk, cov_wk = wls(A_r11[~rej_r11], m_r11[~rej_r11], em_r11[~rej_r11])  # WLS on the stars R11 kept
chi2_wk = np.sum(((m_r11[~rej_r11] - A_r11[~rej_r11] @ th_wk) / em_r11[~rej_r11])**2)

def plot_r11_residuals():
    fig, (ax, ar) = plt.subplots(2, 1, figsize=(7, 4.6), sharex=True, gridspec_kw=dict(height_ratios=[2.2, 1.3], hspace=0.05))
    ax.errorbar(logP_r11[~rej_r11], m_r11[~rej_r11], yerr=em_r11[~rej_r11], fmt='o', color='C0', ms=3, elinewidth=0.6, alpha=0.7, label=f'{(~rej_r11).sum()} kept by Riess et al.')
    ax.errorbar(logP_r11[rej_r11], m_r11[rej_r11], yerr=em_r11[rej_r11], fmt='s', color='C3', ms=4, elinewidth=0.6, alpha=0.8, label=f'{rej_r11.sum()} they rejected (blends, wrong periods)')
    xx = np.linspace(logP_r11.min() - 0.05, logP_r11.max() + 0.05, 20)
    ax.plot(xx, th_w[0] + th_w[1] * (xx - 1), 'C3', lw=1.4, ls='--', label='WLS, all 166')
    ax.invert_yaxis(); ax.set_ylabel('$m$'); ax.legend(frameon=False, fontsize=8, loc='lower left')
    ar.axhline(0, color='k', lw=0.6)
    for k in (-3, 3):
        ar.axhline(k, color='0.6', lw=0.8, ls=':')
    ar.plot(logP_r11[~rej_r11], z_r11[~rej_r11], 'o', color='C0', ms=3)
    ar.plot(logP_r11[rej_r11], z_r11[rej_r11], 's', color='C3', ms=4)
    ar.set_ylabel(r'residual / $\sigma_i$'); ar.set_xlabel(r'$\log_{10} P$ [days]')
    plt.show()
    print(f"{(np.abs(z_r11) > 3).sum()} of 166 stars sit more than 3 sigma from the WLS line; {rej_r11.sum()} carry Riess et al.'s 'rej' flag")
    print(f"WLS on the {(~rej_r11).sum()} kept stars: slope {th_wk[1]:.2f} +/- {np.sqrt(cov_wk[1, 1]):.2f}; chi2 = {chi2_wk:.0f} for {(~rej_r11).sum() - 2} dof, reduced chi2 = {chi2_wk / ((~rej_r11).sum() - 2):.2f}")
    print(f"the {rej_r11.sum()} rejected stars alone carry chi2 = {np.sum(z_r11[rej_r11]**2):.0f} of the {chi2_w:.0f}")

# the mixture model, fit on all 166 by numerical minimization (line + f, Y_b, V_b)
def _nll_mix(p):
    b, a, f, Yb, logVb = p
    if not (0 < f < 1) or logVb < -6 or logVb > 6:
        return 1e30
    mu = A_r11 @ np.array([b, a]); Vb = np.exp(logVb)
    L_in = (1 - f) * np.exp(-0.5 * ((m_r11 - mu) / em_r11)**2) / (np.sqrt(2 * np.pi) * em_r11)
    L_out = f * np.exp(-0.5 * (m_r11 - Yb)**2 / (Vb + em_r11**2)) / np.sqrt(2 * np.pi * (Vb + em_r11**2))
    return -np.sum(np.log(L_in + L_out))

_p0 = [th_w[0], th_w[1], 0.3, np.mean(m_r11), 0.0]
res_mix = minimize(_nll_mix, _p0, method='Nelder-Mead', options=dict(maxiter=20000, xatol=1e-7, fatol=1e-7))
res_mix = minimize(_nll_mix, res_mix.x, method='Nelder-Mead', options=dict(maxiter=20000, xatol=1e-8, fatol=1e-8))
b_mix, a_mix, f_mix, Yb_mix, logVb_mix = res_mix.x
_mu = A_r11 @ np.array([b_mix, a_mix]); _Vb = np.exp(logVb_mix)
_Lin = (1 - f_mix) * np.exp(-0.5 * ((m_r11 - _mu) / em_r11)**2) / em_r11
_Lout = f_mix * np.exp(-0.5 * (m_r11 - Yb_mix)**2 / (_Vb + em_r11**2)) / np.sqrt(_Vb + em_r11**2)
p_out = _Lout / (_Lin + _Lout)                                            # posterior probability each star is an outlier
# error bar on the mixture slope from the numerical Hessian of -ln L (Days 5 and 6), the two line parameters only
def _hess2(fun, x, h=1e-4):
    n = len(x); H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            e_i = np.zeros(n); e_j = np.zeros(n); e_i[i] = h; e_j[j] = h
            H[i, j] = (fun(x + e_i + e_j) - fun(x + e_i - e_j) - fun(x - e_i + e_j) + fun(x - e_i - e_j)) / (4 * h * h)
    return H
_H = _hess2(_nll_mix, res_mix.x, h=np.array([1e-3, 1e-3, 1e-3, 1e-3, 1e-3]).mean())
try:
    sig_a_mix = np.sqrt(np.linalg.inv(_H)[1, 1])
except np.linalg.LinAlgError:
    sig_a_mix = np.nan

# the same mixture with last week's intrinsic scatter as a 6th parameter: variance sigma_i^2 + sigma_int^2 on the line
def _nll_mix_int(p):
    b, a, f, Yb, logVb, logs = p
    if not (0 < f < 1) or abs(logVb) > 6 or logs < -6 or logs > 1:
        return 1e30
    mu = A_r11 @ np.array([b, a]); Vb = np.exp(logVb); s2 = em_r11**2 + np.exp(2 * logs)
    L_in = (1 - f) * np.exp(-0.5 * (m_r11 - mu)**2 / s2) / np.sqrt(2 * np.pi * s2)
    L_out = f * np.exp(-0.5 * (m_r11 - Yb)**2 / (Vb + em_r11**2)) / np.sqrt(2 * np.pi * (Vb + em_r11**2))
    return -np.sum(np.log(L_in + L_out))

res_mix_int = None
for _a0 in (-3.0, -3.5):
    for _ls0 in (-2.0, -1.0):
        _r = minimize(_nll_mix_int, [th_w[0], _a0, 0.2, np.mean(m_r11), 0.0, _ls0], method='Nelder-Mead',
                      options=dict(maxiter=40000, xatol=1e-8, fatol=1e-8))
        if res_mix_int is None or _r.fun < res_mix_int.fun:
            res_mix_int = _r
a_mix_int, f_mix_int, s_int_mix = res_mix_int.x[1], res_mix_int.x[2], np.exp(res_mix_int.x[5])
_H2 = _hess2(_nll_mix_int, res_mix_int.x, h=1e-3)
try:
    sig_a_mix_int = np.sqrt(np.linalg.inv(_H2)[1, 1])
except np.linalg.LinAlgError:
    sig_a_mix_int = np.nan

def plot_r11_mixture():
    fig, ax = plt.subplots(figsize=(7, 4.2))
    xx = np.linspace(logP_r11.min() - 0.05, logP_r11.max() + 0.05, 20)
    sc = ax.scatter(logP_r11, m_r11, c=p_out, cmap='coolwarm', vmin=0, vmax=1, s=18, zorder=3)
    ax.errorbar(logP_r11, m_r11, yerr=em_r11, fmt='none', ecolor='0.7', elinewidth=0.5, zorder=2)
    ax.plot(xx, b_mix + a_mix * (xx - 1), 'k', lw=1.8, label=fr'mixture model: slope {a_mix:.2f} $\pm$ {sig_a_mix:.2f}, $f$ = {f_mix:.2f}')
    ax.plot(xx, th_w[0] + th_w[1] * (xx - 1), 'C3', lw=1.2, ls='--', label=fr'WLS, all 166: slope {th_w[1]:.2f} $\pm$ {np.sqrt(cov_w[1, 1]):.2f}')
    ax.plot(xx, th_wk[0] + th_wk[1] * (xx - 1), 'C0', lw=1.2, ls=':', label=fr"WLS on Riess et al.'s {(~rej_r11).sum()}: slope {th_wk[1]:.2f} $\pm$ {np.sqrt(cov_wk[1, 1]):.2f}")
    cb = plt.colorbar(sc, ax=ax, pad=0.01); cb.set_label('probability the star is an outlier', fontsize=9)
    ax.invert_yaxis(); ax.set_xlabel(r'$\log_{10} P$ [days]'); ax.set_ylabel('$m$')
    ax.legend(frameon=False, fontsize=8, loc='lower left')
    plt.tight_layout()
    plt.show()
    n_flag = (p_out > 0.5).sum(); agree = ((p_out > 0.5) & rej_r11).sum()
    print(f"mixture: slope {a_mix:.3f} +/- {sig_a_mix:.3f}, outlier fraction f = {f_mix:.2f}, background mean {Yb_mix:.2f} mag, width sqrt(V_b) = {np.sqrt(_Vb):.2f} mag")
    print(f"{n_flag} stars have p(outlier) > 0.5; {agree} of them are on Riess et al.'s rejected list of {rej_r11.sum()} (no threshold was chosen by us)")
    print(f"with sigma_int as a 6th parameter: slope {a_mix_int:.2f} +/- {sig_a_mix_int:.2f}, f = {f_mix_int:.2f}, sigma_int = {s_int_mix:.2f} mag; -ln L drops from {res_mix.fun:.1f} to {res_mix_int.fun:.1f}")

def plot_r11_wls():
    fig, ax = plt.subplots(figsize=(7, 4.2))
    xx = np.linspace(logP_r11.min() - 0.05, logP_r11.max() + 0.05, 20)
    ax.errorbar(logP_r11, m_r11, yerr=em_r11, fmt='o', color='C0', ms=3, elinewidth=0.6, alpha=0.7, label=f'NGC 4258, {m_r11.size} Cepheids (Riess et al. 2011)')
    ax.plot(xx, th_o[0] + th_o[1] * (xx - 1), 'C2', lw=1.6, label=fr'OLS: slope {th_o[1]:.2f} $\pm$ {np.sqrt(cov_o[1, 1]):.2f}')
    ax.plot(xx, th_w[0] + th_w[1] * (xx - 1), 'C3', lw=1.8, label=fr'WLS: slope {th_w[1]:.2f} $\pm$ {np.sqrt(cov_w[1, 1]):.2f}')
    ax.invert_yaxis()
    ax.set_xlabel(r'$\log_{10} P$ [days]'); ax.set_ylabel('$m$ (F160W, colour-corrected)')
    ax.legend(frameon=False, fontsize=9)
    plt.tight_layout()
    plt.show()
    print(f"OLS: slope {th_o[1]:.3f} +/- {np.sqrt(cov_o[1, 1]):.3f} (errors from the scatter, s = {np.sqrt(res_o @ res_o / nu_w):.2f} mag)")
    print(f"WLS: slope {th_w[1]:.3f} +/- {np.sqrt(cov_w[1, 1]):.3f} (errors from the error bars); chi2 = {chi2_w:.0f} for {nu_w} dof, reduced chi2 = {chi2_w / nu_w:.2f}")
    print(f"median error bar {np.median(em_r11):.2f} mag, range {em_r11.min():.2f} to {em_r11.max():.2f}")

## 166 Cepheids in the anchor galaxy

<div style="display:flex; align-items:flex-start; gap:1.2em;">
<img src="images/ngc4258_m106_heic1302a.jpg" alt="Hubble Space Telescope image of the spiral galaxy NGC 4258 (Messier 106), a tilted spiral with a bright yellow core and blue star-forming arms" style="max-height:300px; border-radius:3px;">
<div>

* Hubble's distances came from Cepheids, and today's $H_0$ still does
* the distance ladder is the chain of calibrations from the nearest stars to distant galaxies, one rung at a time
* NGC 4258 is the anchor rung.
* Water clouds orbiting its black hole emit at radio wavelengths (masers), and their orbits give a geometric distance
* so its Cepheids calibrate the period-luminosity relation you fit last week; Riess et al. 2011 measured 166 of them with the Hubble Space Telescope (WFC3)
* $m$ is the near-infrared magnitude (F160W, 1.6 $\mu$m, where dust matters least) with a colour term $-0.41\,(V - I)$ subtracted for each star's temperature and reddening
* one galaxy, one distance, and apparent $m$ is fine.
* The $x$ axis is pivoted at $P = 10$ d so slope and intercept are not correlated
* every star has its own error bar, 0.1 to 0.9 mag.
* Now the weights matter

</div>
</div>

<div style="font-size:0.7em; color:#666;">Image: NASA, ESA, the Hubble Heritage Team (STScI/AURA), and R. Gendler (for the Hubble Heritage Team). Acknowledgment: J. GaBany. ESA/Hubble heic1302a, CC BY 4.0.</div>


In [ ]:
# 166 Cepheids in NGC 4258: OLS and WLS lines, each with its slope error bar
plot_r11_wls()

* the slopes disagree by 2 of OLS's error bars (5 of WLS's).
* The faint, noisy stars drag the unweighted line
* reduced $\chi^2 = 6.6$ for 164 degrees of freedom.
* Either the error bars are wrong, or the model is, and the residuals will tell us which
* with equal error bars (the $\sigma = 0.06$ mag we assumed for the AAVSO Cepheids last week) OLS and WLS are the same fit!


## Which stars? The residuals say

*Hubble's problem 4 of 5: some of the points are wrong.*

* his own data had them too. His distances to the nearest galaxies were off by far more than the calibration alone explains
* Let's plot each star's residual from the WLS line in units of its own error bar, the normalized residual
* Gaussian noise puts 99.7% of points inside $\pm 3$


In [ ]:
# the WLS fit on the 166 Cepheids, and each star's residual in units of its own error bar; Riess et al.'s own rejects marked
plot_r11_residuals()


* 16 of the 166 sit more than 3$\sigma$ off, some by 10 or more.
* Blends (2 stars in 1 pixel look like 1 brighter star) and wrong periods
* Riess et al. flagged 48 in their Table 2 (more than 2.5$\sigma$ or 0.75 mag off, clipped and refit to convergence, which catches more than our one pass)
* of the total $\chi^2$ of 1089, those 48 contribute 975.
* Drop them and reduced $\chi^2$ is 0.96! The error bars were fine, the Gaussian assumption was not
* Real data has outliers, and least squares gives every point full weight!
* a least-squares fit is a weighted mean (for a single column, $\hat\theta = \sum w_i y_i / \sum w_i$ with $w_i = 1/\sigma_i^2$), and Day 2 showed what one outlier does to a mean
* 2 answers: clip them (what Riess et al. did; $k\sigma$ clipping is ad hoc, the threshold is yours), or use a robust estimator that does not care about them, like Theil-Sen from Day 3
* or model them, which is Lab 01's Part 2 and the next slide


## Outliers: the mixture model

Let's write the likelihood out. It is a mixture between a signal and a background, the same model as Day 3 with a new mean for the signal (the line):

$$\mathcal{L}_i = (1 - f)\,\mathcal{N}\!\left(y_i \mid A_i\boldsymbol{\theta},\, \sigma_i\right) + f\,\mathcal{N}\!\left(y_i \mid Y_b,\, \sqrt{V_b + \sigma_i^2}\right)$$

* the signal sits on the line with its own $\sigma_i$, probability $1 - f$
* the background is a broad Gaussian of mean $Y_b$, probability $f$
* its width is $\sqrt{V_b + \sigma_i^2}$, because the bad points have their measurement error plus an extra spread $V_b$ we fit (independent variances add)
* we fit it to all 166 Cepheids, numerically, as last week.
* 5 parameters, every star kept


In [ ]:
# the mixture model fit to all 166 Cepheids: the line, and each star coloured by the probability that it is an outlier
plot_r11_mixture()


* no threshold was chosen.
* The data set $f = 0.27$, and 23 of the 27 stars with $p(\textrm{outlier}) > 0.5$ are on Riess et al.'s list
* but the slope is $-3.62 \pm 0.17$, against $-3.08 \pm 0.11$ for WLS on their 118 kept stars (the dotted line). 2.7$\sigma$ apart!
* the line has no intrinsic scatter, and the 5-parameter model labels the good stars' own scatter as outliers
* add last week's $\sigma_{\textrm{int}}$ as a 6th parameter: slope $-3.39 \pm 0.17$, $f = 0.07$, $\sigma_{\textrm{int}} = 0.44$ mag, and $\Delta\chi^2 = 50$ for 1 extra parameter (the test from last class says decisive)
* the likelihood picks the best of the models you offer, not the truth.
* Two decompositions of the same 166 stars, two slopes.
* Thursday's theme: write down how the data were made


## Nuisance parameters

<img src="images/hogg2010_fig4_mixture_fit.png" alt="Hogg, Bovy and Lang 2010 Figure 4: twenty points with error bars, four outliers sitting off the line, and the mixture-model straight line passing through the sixteen good points" style="display:block;margin:0 auto;max-height:230px">

<div style="font-size:0.8em; text-align:center; color:#666;">Hogg, Bovy &amp; Lang 2010, arXiv:1008.4686, Fig. 4: the same mixture model on Lab 01's 20 points, 4 of them deliberately bad.</div>

* $f$, $Y_b$ and $V_b$ are called **nuisance parameters**: fitted alongside the line, then integrated over, and left out of the result.
* Here we took their best-fit values
* this is Lab 01's Part 2 (Hogg, Bovy &amp; Lang 2010, above), and Day 3's RV drift with its outliers
* Lab 02 was the same model with a star instead of a line: $f_{\textrm{bg}}$ times a flat background plus $(1 - f_{\textrm{bg}})$ times a Moffat
* the other fix is heavier tails, a Student-t likelihood instead of a second Gaussian
* Student-t is a family of bell curves with fatter tails than a Gaussian; Day 2's Cauchy is the extreme member

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Same mixture model as Day 3 and Labs 01 and 02, with a line for the signal.**

</div>


## Which variable has the error?

*Hubble's problem 5 of 5, and the one Thursday takes up: the noisy variable was $x$.*

* outliers broke the Gaussian assumption.
* Assumption 2 was that $x$ is exact
* least squares minimizes the vertical residuals, i.e. it assumes all the noise is in $y$
* Hubble's distance calibration was off by a constant factor, and that just rescales $K$
* they also scattered randomly from galaxy to galaxy, and that random error is what least squares gets wrong
* why? Scatter in $r$ spreads the points out horizontally while the $v$'s stay put, so the slope $\Delta v / \Delta r$ of the best-fit line drops
* Let's regress the other way, $r$ on $v$, and compare


In [ ]:
# OLS of v on r, and OLS of r on v, on the same axes (a "skip" cell)
A_rv = np.vstack([np.ones_like(v_hub), v_hub]).T
theta_rv, cov_rv, _ = ols(A_rv, r_hub)          # r = r0 + v / K'
K_inv = 1.0 / theta_rv[1]                        # r = r0 + v / K'  ->  v = K' (r - r0)
r0_inv = theta_rv[0]

def plot_two_regressions():
    fig, ax = plt.subplots(figsize=(7, 4.2))
    rr = np.linspace(-0.1, 2.1, 20)
    ax.plot(r_hub, v_hub, 'o', color='C0', ms=6)
    ax.plot(rr, v0_hat + K_hat * rr, 'C3', lw=1.8, label=fr'$v$ on $r$ (vertical misses): $K$ = {K_hat:.0f}')
    ax.plot(rr, K_inv * (rr - r0_inv), 'C2', lw=1.8, label=fr'$r$ on $v$ (horizontal misses): $K$ = {K_inv:.0f}')
    ax.set_xlabel('distance $r$ [Mpc]'); ax.set_ylabel('$v$ [km/s]')
    ax.legend(frameon=False, fontsize=9)
    plt.tight_layout()
    plt.show()
    print(f"v on r: K = {K_hat:.0f} km/s/Mpc;  r on v: K = {K_inv:.0f} km/s/Mpc;  ratio {K_inv / K_hat:.2f}")

In [ ]:
# the same 24 points, two least-squares lines, two slopes
plot_two_regressions()

* the two slopes differ by 60%!
* Each answers a different question: the first predicts $v$ from $r$, the second $r$ from $v$
* random scatter in $r$ drags the $v$-on-$r$ slope toward 0.
* Since the distances were the noisy variable, $r$ on $v$ is the better-posed fit here
* NB the factor of 7 is the calibration, a separate error

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Regress the noisy variable on the exact one. If both are noisy, neither line recovers the relation (Thursday).**

</div>


## Five lines through one cloud

<img src="images/isobe1990_five_regression_lines.png" alt="Isobe et al. 1990 figure: one scatter of points with five straight lines through it, ordinary least squares of Y on X, of X on Y, their bisector, the orthogonal regression line and the reduced major axis, fanning out around the centroid" style="display:block;margin:0 auto;max-height:330px">

<div style="font-size:0.8em; text-align:center; color:#666;">Isobe, Feigelson, Akritas &amp; Babu 1990, ApJ 364, 104, Fig. 2: Schechter's 1980 Faber-Jackson data. Single cited figure, non-commercial course use.</div>

* a real scaling relation (one galaxy property against another, in log-log): Faber-Jackson, an elliptical's luminosity against its velocity dispersion
* velocity dispersion is the spread in its stars' speeds, and both axes are measured with error
* 5 least-squares lines: OLS($Y|X$), OLS($X|Y$), their bisector, the orthogonal fit (perpendicular misses), the reduced major axis (geometric mean of the 2 OLS slopes)
* all 5 pass through the centroid and differ in how they split the scatter. They agree only when the scatter is small
* Isobe et al. surveyed the literature and found that most papers picked one without saying which
* they recommend OLS($Y|X$) for predicting $Y$, the bisector for the underlying relation.
* Thursday shows why even that fails once both variables are noisy

## Beyond least squares: errors in $x$, upper limits

<img src="images/colombo2025_fig9_resolved_sfr_mmol.png" alt="Colombo et al. 2025 Figure 9: star-formation rate against molecular-gas mass for galaxies in nine panels, detections as circles with error bars, CO non-detections as leftward upper-limit arrows, and the fitted regression lines" style="display:block;margin:0 auto;max-height:320px">

<div style="font-size:0.8em; text-align:center; color:#666;">Colombo et al. 2025, A&amp;A 699, A367 (arXiv:2507.06406; T. Wong and K. D. French co-authors), Fig. 9, CC BY 4.0: SFR against molecular-gas mass with CO non-detections as upper limits, fit with linmix.</div>

* a scaling relation from this building: how fast a galaxy forms stars (SFR, solar masses per year) against how much cold molecular gas it holds
* the gas mass comes from the CO line, the recap's molecule
* error bars on both axes, and galaxies with no CO detection at all, which only give an upper limit (the arrows).
* A diagonal $\Sigma$ describes none of that
* the standard tool is called Kelly's method (Kelly 2007): the likelihood of each $(x_i, y_i)$ pair given a line, intrinsic scatter, and errors on both axes
* the Python package is `linmix`, and Thursday derives it


## Predicting the lag: Yue Shen's fit

<img src="images/shen2024_fig11_hbeta_crop.png" alt="Shen et al. 2024 Figure 11, H-beta panel: rest-frame emission-line lag against host-corrected 5100 angstrom luminosity for SDSS reverberation-mapping quasars, with error bars, light-blue posterior draws of the fitted relation, and the Bentz et al. 2013 relation dashed" style="display:block;margin:0 auto;max-height:360px">

<div style="font-size:0.8em; text-align:center; color:#666;">Shen et al. 2024, ApJS 272, 26 (arXiv:2305.01014, CC BY 4.0), Fig. 11: the H&beta; radius-luminosity relation for SDSS reverberation-mapping (SDSS-RM) quasars, fit with Kelly 2007's regression.</div>

* in reverberation mapping a quasar's emission line brightens after its continuum does, delayed by the light-travel time $\tau$ across the gas
* so $c\tau$ is the size of the emitting region, and with a velocity that gives the black hole's mass
* the predict-$Y$ case, also from this building.
* Yue Shen's group fits lag against luminosity $L$ to predict $\tau$ for quasars nobody can monitor for years
* both axes have error bars and the relation has real scatter, hence Kelly's method (Thursday)

## Today, 3:45 pm, in this room: Charlotte Ward (Penn State)

<div style="display:flex; align-items:flex-start; gap:1.4em; margin-top:0.4em;">
<img src="images/speaker_charlotte_ward.jpg" alt="Portrait of astronomer Charlotte Ward" style="max-height:400px; border-radius:4px;">
<div>

* **"Better Together: Combining the Strengths of Rubin, Euclid, and Roman for Time-domain Science and Cosmology"**
* the double-peaked line from last class, 250 times over.
* Ward et al. 2024 found 250 in ZTF (the Zwicky Transient Facility, which images the northern sky every 2 nights)
* of 12 re-observed, half changed which peak is stronger over 10 to 20 years
* Thursday's lab is built on her sample (Ward et al. 2022): black holes in dwarf galaxies, the small end of the black-hole-mass scaling relation
* **+5 points on Lab 04 if you go**, same deal as the El-Badry talk: at least a page of notes including 2 questions you had, as a text file in `submissions/<netid>/`, due Noon Tue Sep 29
* refreshments 3:30 pm, Astronomy 222.
* Colloquium is every Tuesday at 3:45, right here, and you should be at it

</div>
</div>

<div style="font-size:0.7em; color:#666;">Photo: charlotteaward.github.io, used to advertise her own talk. Ward et al. 2024, ApJ 961, 172; Ward et al. 2022, ApJ 936, 104 (arXiv:2110.13098). Listing: astro.illinois.edu/news-events/astrophysics-colloquium.</div>

## Wrapping up: Hubble's one line, five problems

1. least squares is the Gaussian likelihood with noise in $y$ only, and the MLE has a closed form
2. linear means linear in the parameters (Hubble's 4 columns), so polynomials, sinusoids and templates all use the same normal equations. OLS error bars come from the scatter, WLS error bars from your $\sigma_i$ (the inverse Fisher information)
3. real error bars make the weights matter, and a reduced $\chi^2$ far from 1 means the error bars or the model are wrong
4. for outliers, normalized residuals first, then the mixture model from Day 3. The likelihood picks among the models you offer, so the model for the bad points has to be right too
5. which variable you regress on which says where the noise is, and it changes the slope

* the 3rd assumption is independence.
* Correlated errors put off-diagonal terms in $\Sigma$, which is called generalized least squares (GLS): Thursday
* Thu Sep 24: GLS, noise in $x$ too, why every OLS slope is then biased low, and the generative fix (a model for how the data were produced). Lab 04 posts


## Before you go

* **Lab 03** is due tomorrow, Wed Sep 23, by Noon (fork then PR, as always); every date is in `ASSIGNMENTS.md`
* **Lab 04** posts Thursday: a black-hole mass scaling relation, fit 4 ways, and only one of them is unbiased
* Nothing from today is collected.
* Quiz 2 comes back once graded

## Coming up

* Colloquium **today, 3:45, this room**: Charlotte Ward (Penn State), "Better Together: Combining the Strengths of Rubin, Euclid, and Roman for Time-domain Science and Cosmology"
* Next week: David Charbonneau (Harvard), colloquium Tue Sep 29, 3:45, here
* his public Iben lecture, "The Terrestrial Worlds of Other Stars", is Wed Sep 30 in Lincoln Hall Theater
* Thu Sep 24: correlated errors, errors in both variables, and the generative answer
* Tue Sep 29 (on Zoom): priors, posteriors, the negative-parallax star, and the Bayes factor